# Chapter 25 — From Deltas to Operators

**Book alignment:** Embeddings From First Principles, Chapter 25

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** Put the difference vector on a ladder — identity, delta,
rank-1, low-rank, affine, local, nonlinear. For typed sentence-embedding edits on RELATE-DOC
(Wave 5), does anything above a **constant offset** ever win — or are the expressive rungs
(full-affine, local-affine, MLP) strictly worse on every transformation?

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    return json.loads((EXP / wave / "artifacts" / f"{name}.json").read_text())

## 1. The operator bake-off (Wave 5)

In [ ]:
bk = art("wave5", "operator-bakeoff")
bar = bk["pass_bar"]
simple_ops     = ("identity", "constant_delta", "rank1_affine", "lowrank_affine")
expressive_ops = ("full_affine", "local_affine", "mlp")
print(f"pass bar: reconstruction cosine >= {bar}\n")
print(f"{'transformation':22} {'identity':>9} {'delta':>7} {'affine':>7} {'mlp':>7}   simplest passing")
for name, t in bk["per_transformation"].items():
    ops = t["operators"]
    order = ("identity", "constant_delta", "rank1_affine", "lowrank_affine",
             "full_affine", "local_affine", "mlp")
    passing = [k for k in order if k in ops and ops[k]["reconstruction_cos"] >= bar]
    print(f"{name:22} {ops['identity']['reconstruction_cos']:>9.2f}"
          f" {ops['constant_delta']['reconstruction_cos']:>7.2f}"
          f" {ops['full_affine']['reconstruction_cos']:>7.2f}"
          f" {ops['mlp']['reconstruction_cos']:>7.2f}   {passing[0] if passing else 'NONE PASS'}")

In [ ]:
pt = bk["per_transformation"]
# grammatical / role / tense / relation-direction edits: IDENTITY passes - the embedding barely moves
for name in ("active_to_passive", "present_to_past", "relation_swap"):
    assert pt[name]["operators"]["identity"]["reconstruction_cos"] >= bar

# register / length / negation edits: NOTHING on the ladder passes
for name in ("formal_to_informal", "statement_to_negation", "verbose_to_concise"):
    best = max(o["reconstruction_cos"] for o in pt[name]["operators"].values())
    assert best < bar

# on EVERY transformation the best expressive rung loses to the best simple rung
for name, t in pt.items():
    ops = t["operators"]
    best_simple     = max(ops[k]["reconstruction_cos"] for k in simple_ops if k in ops)
    best_expressive = max(ops[k]["reconstruction_cos"] for k in expressive_ops if k in ops)
    assert best_simple > best_expressive, name

# a full affine map fit on 10-45 pairs lands on the far side of the sphere
assert pt["active_to_passive"]["operators"]["full_affine"]["reconstruction_cos"] < 0.6

# inverse consistency: round trips lose most of the signal
ic = bk["inverse_consistency"]["relation_swap"]
print(f"relation_swap: single forward cos {ic['single_forward_cos']:.2f},"
      f" T^-1(T(x)) returns to {ic['inverse_consistency_cos']:.2f}")
assert ic["inverse_consistency_cos"] < ic["single_forward_cos"]
print("\nthe delta/analogy tradition assumed a reusable middle direction; RELATE-DOC shows a cliff, not a slope")

## What we earned

The operator ladder is `T(x) = Wx + b` — a delta is `W = I`, a linear bridge is `b = 0`, so
alignment and semantic editing are one ladder. Measured on typed sentence-embedding edits:
grammatical, role, tense and relation-direction edits are **identity** — the embedding
barely moves ("Helios acquired Pine" and "Pine acquired Helios" reconstruct at cos ≈ 0.98
with no operator at all). A few edits (claim-weakening, temporal shifts) are captured by a
**constant offset**. Register, length and negation edits genuinely move the embedding and
**no operator on the ladder recovers the move** — and on every transformation the expressive
rungs (full-affine, local-affine, MLP) are strictly worse than the simple ones. The graded
middle was not observed.

**Notebook 24 / Chapter 24** composes every artifact into one runtime — the Embedding
Observatory.